# 01 Tokenization：BPE 分词——从字符到子词

> 前置：`01-linear-algebra/01`（向量与查找表）。
> 目标：理解为什么 LLM 不直接吃字符/单词，而是用**子词（subword）**；完整实现 BPE（Byte Pair Encoding）的训练与编解码。

## 为什么需要子词

| 方案 | 优点 | 致命缺点 |
|---|---|---|
| 字符级 | 词表极小（~70） | 序列太长、计算浪费 |
| 单词级 | 序列短 | 词表无限（新词/拼写变体/形态变化爆炸） |
| **子词级** | 词表固定且中等（3万~20万） | —— |

子词把词拆成**可复用的最小单位**：`low` + `est` = `lowest`，`low` + `er` = `lower`。模型看到 `est` 就"知道"比较级，不用重新学。

## BPE 算法（两步）

1. **训练**（统计合并）：
   - 初始词表 = 全部字符；
   - 反复统计相邻符号对的频次，把**最高频的一对**合并成一个新符号，直到词表达到目标大小。
2. **编码**：把词按训练出的合并顺序，贪婪地应用合并。

> GPT-2/3/4 用的是 BPE 的变体（GPT-2 基于字节、GPT-4 的 tiktoken 合并了更细规则），但核心就是"最高频相邻对合并"。

## 经典语料（BPE 论文原例）

`low, lower, newest, widest, lowest, newer, new, wide, lowered`——含 `low/lowest/lower`、`new/newer/newest`、`wide/widest` 三组形态家族，是观察合并过程的完美素材。

In [1]:
from collections import Counter

corpus = ["low"]*5 + ["lower"]*2 + ["newest"]*6 + ["widest"]*3 + ["lowest"]*4 + ["newer"]*2 + ["new"]*8 + ["wide"]*3 + ["lowered"]*1
freq = Counter(corpus)
splits = {w: list(w) for w in freq}

print("词频:", dict(freq))
print("初始词表（字符）:", sorted({c for w in splits for c in w}))
print("初始词表大小:", len({c for w in splits for c in w}))

词频: {'low': 5, 'lower': 2, 'newest': 6, 'widest': 3, 'lowest': 4, 'newer': 2, 'new': 8, 'wide': 3, 'lowered': 1}
初始词表（字符）: ['d', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
初始词表大小: 10


## 训练：迭代合并最高频相邻对

每一轮：
1. 统计所有相邻对的出现次数（按词频加权）；
2. 找最高频对 `(a, b)`，合并成新符号 `ab`；
3. 更新所有词的切分。

In [2]:
merges = []   # (新符号, 频次, a, b)
for step in range(16):
    pair_counts = Counter()
    for w, s in splits.items():
        for pair in zip(s, s[1:]):
            pair_counts[pair] += freq[w]
    if not pair_counts:
        break
    (a, b), c = pair_counts.most_common(1)[0]
    merges.append((a + b, c, a, b))
    new_splits = {}
    for w, s in splits.items():
        ns, i = [], 0
        while i < len(s):
            if i + 1 < len(s) and s[i] == a and s[i + 1] == b:
                ns.append(a + b); i += 2
            else:
                ns.append(s[i]); i += 1
        new_splits[w] = ns
    splits = new_splits

print("前 12 次合并（符号, 频次, 由哪两个合并）:")
for m, c, a, b in merges[:12]:
    print(f"  {a!r}+{b!r} → {m!r}   (频次 {c})")

前 12 次合并（符号, 频次, 由哪两个合并）:
  'n'+'e' → 'ne'   (频次 16)
  'ne'+'w' → 'new'   (频次 16)
  'e'+'s' → 'es'   (频次 13)
  'es'+'t' → 'est'   (频次 13)
  'l'+'o' → 'lo'   (频次 12)
  'lo'+'w' → 'low'   (频次 12)
  'new'+'est' → 'newest'   (频次 6)
  'w'+'i' → 'wi'   (频次 6)
  'wi'+'d' → 'wid'   (频次 6)
  'e'+'r' → 'er'   (频次 5)
  'low'+'est' → 'lowest'   (频次 4)
  'low'+'er' → 'lower'   (频次 3)


In [3]:
final_vocab = sorted({c for s in splits.values() for c in s})
print("训练后词表:", final_vocab)
print(f"词表大小: {len({c for w in corpus for c in w})} → {len(final_vocab)}")

print("\n各词最终切分:")
for w in sorted(freq):
    print(f"  {w:8s} → {splits[w]}")

训练后词表: ['d', 'low', 'lower', 'lowere', 'lowest', 'new', 'newer', 'newest', 'wide', 'widest']
词表大小: 10 → 10

各词最终切分:
  low      → ['low']
  lower    → ['lower']
  lowered  → ['lowere', 'd']
  lowest   → ['lowest']
  new      → ['new']
  newer    → ['newer']
  newest   → ['newest']
  wide     → ['wide']
  widest   → ['widest']


## 编码：按合并顺序贪心应用

对**没见过的词** `lowered` 之外的变体，如 `lowly`、`newer`，按训练出的合并顺序（越早越"根本"）逐对应用。

In [4]:
def encode(word, merge_order):
    s = list(word)
    for tok, _, a, b in merge_order:
        ns, i = [], 0
        while i < len(s):
            if i + 1 < len(s) and s[i] == a and s[i + 1] == b:
                ns.append(tok); i += 2
            else:
                ns.append(s[i]); i += 1
        s = ns
    return s

for w in ["lowest", "newer", "lowly", "newerest"]:
    toks = encode(w, merges)
    print(f"{w:10s} → {toks}")

# 压缩率：字符数 vs token 数
chars = sum(len(w) * f for w, f in freq.items())
tokens = sum(len(splits[w]) * f for w, f in freq.items())
print(f"\n语料字符数 = {chars}，切分后 token 数 = {tokens}，压缩比 = {chars / tokens:.2f}x")

lowest     → ['lowest']
newer      → ['newer']
lowly      → ['low', 'l', 'y']
newerest   → ['newer', 'est']

语料字符数 = 156，切分后 token 数 = 35，压缩比 = 4.46x


## 总结与进阶

1. **BPE = 数据压缩思想**：出现越频繁的模式越早合并成单一 token，词表固定后即"压缩"完成。
2. **词表超参数**：GPT-2 用 50257，Llama 用 32000，GPT-4 用 ~100k——越大表示越细、序列越短，但 embedding 矩阵越大。
3. **细节**：真实 BPE 还要处理 ①词边界标记（`</w>`，防止跨词合并）②字节级 fallback（未知 UTF-8 字节）；tiktoken 还做了正则分词（数字/标点拆分）。

## 课后练习

1. 把语料里的 `new` 频次从 8 提到 80，重新训练——观察哪些合并变了顺序？为什么？
2. 编码 `lowered`（训练语料里出现过）和 `lowly`（没见过），比较切分质量，解释差异。
3. 增加合并轮数到 30，观察词表饱和现象（新合并频次骤降）——思考"词表大小怎么定"。